In [6]:
# ==================================================
# 1. IMPORTS
# ==================================================
!pip install scapy xgboost lightgbm
import os
import re
import math
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


from collections import Counter, defaultdict

from scapy.all import PcapReader, IP, IPv6, TCP, UDP

from sklearn.model_selection import GroupShuffleSplit, RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

from sklearn.ensemble import RandomForestClassifier

# Try importing gradient boosters
try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False


# ==================================================
# 2. CONFIGURATION
# ==================================================

RANDOM_SEED = 42

DATASET_PATH = "USTC-TFC2016"

plt.style.use("ggplot")

print("Imports completed.")

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 22.3.1 -> 26.2.1
[notice] To update, run: C:\Program Files\Python310\python.exe -m pip install --upgrade pip


ModuleNotFoundError: No module named 'scapy'

In [ ]:
!pip install -q scapy ja3

In [ ]:
import os

extract_path = "/content/USTC-TFC2016"
os.makedirs(extract_path, exist_ok=True)

!unzip -q /content/USTC-TFC2016-master.zip -d /content/USTC-TFC2016

print("Dataset extracted!")

replace /content/USTC-TFC2016/USTC-TFC2016-master/Benign/BitTorrent.pcap? [y]es, [n]o, [A]ll, [N]one, [r]ename: Dataset extracted!


In [ ]:
# ==================================================
# 4. EXTRACT ALL 7Z FILES
#    PRESERVE BENIGN / MALWARE LABEL
# ==================================================

!apt-get install -y p7zip-full -qq

import os
import glob
import shutil
import subprocess

dataset_root = "/content/USTC-TFC2016/USTC-TFC2016-master"
extract_root = "/content/USTC_extracted"

# Remove old extraction
if os.path.exists(extract_root):
    shutil.rmtree(extract_root)

os.makedirs(extract_root, exist_ok=True)


for label_name in ["Benign", "Malware"]:

    source_dir = os.path.join(
        dataset_root,
        label_name
    )

    seven_zip_files = glob.glob(
        os.path.join(
            source_dir,
            "*.7z"
        )
    )

    for archive in seven_zip_files:

        archive_name = os.path.splitext(
            os.path.basename(archive)
        )[0]

        output_dir = os.path.join(
            extract_root,
            label_name,
            archive_name
        )

        os.makedirs(
            output_dir,
            exist_ok=True
        )

        print(
            f"Extracting [{label_name}] "
            f"{os.path.basename(archive)}"
        )

        subprocess.run(
            [
                "7z",
                "x",
                archive,
                f"-o{output_dir}",
                "-y"
            ],
            check=True,
            stdout=subprocess.DEVNULL
        )


print("\n✅ All 7z files extracted.")

Extracting [Benign] Weibo.7z
Extracting [Benign] SMB.7z
Extracting [Malware] Neris.7z
Extracting [Malware] Nsis-ay.7z
Extracting [Malware] Htbot.7z
Extracting [Malware] Virut.7z
Extracting [Malware] Shifu.7z
Extracting [Malware] Cridex.7z
Extracting [Malware] Geodo.7z

✅ All 7z files extracted.


In [ ]:
# ==================================================
# 5. FIND ALL PCAPS
# ==================================================

import glob
import os

all_pcaps = []

# Original PCAPs
all_pcaps += glob.glob(
    os.path.join(
        dataset_root,
        "**",
        "*.pcap"
    ),
    recursive=True
)

# Extracted PCAPs
all_pcaps += glob.glob(
    os.path.join(
        extract_root,
        "**",
        "*.pcap"
    ),
    recursive=True
)

all_pcaps = sorted(
    set(
        os.path.abspath(p)
        for p in all_pcaps
    )
)

print(
    "Total PCAP files found:",
    len(all_pcaps)
)

for p in all_pcaps:
    print(p)

Total PCAP files found: 24
/content/USTC-TFC2016/USTC-TFC2016-master/Benign/BitTorrent.pcap
/content/USTC-TFC2016/USTC-TFC2016-master/Benign/FTP.pcap
/content/USTC-TFC2016/USTC-TFC2016-master/Benign/Facetime.pcap
/content/USTC-TFC2016/USTC-TFC2016-master/Benign/Gmail.pcap
/content/USTC-TFC2016/USTC-TFC2016-master/Benign/MySQL.pcap
/content/USTC-TFC2016/USTC-TFC2016-master/Benign/Outlook.pcap
/content/USTC-TFC2016/USTC-TFC2016-master/Benign/Skype.pcap
/content/USTC-TFC2016/USTC-TFC2016-master/Benign/WorldOfWarcraft.pcap
/content/USTC-TFC2016/USTC-TFC2016-master/Malware/Miuref.pcap
/content/USTC-TFC2016/USTC-TFC2016-master/Malware/Tinba.pcap
/content/USTC-TFC2016/USTC-TFC2016-master/Malware/Zeus.pcap
/content/USTC_extracted/Benign/SMB/SMB/SMB-1.pcap
/content/USTC_extracted/Benign/SMB/SMB/SMB-2.pcap
/content/USTC_extracted/Benign/Weibo/Weibo/Weibo-1.pcap
/content/USTC_extracted/Benign/Weibo/Weibo/Weibo-2.pcap
/content/USTC_extracted/Benign/Weibo/Weibo/Weibo-3.pcap
/content/USTC_extracted/

In [ ]:
!unzip -t /content/USTC-TFC2016-master.zip

Archive:  /content/USTC-TFC2016-master.zip
4bc9683b996f582c3843815b68da8e4dce9c7e1e
    testing: USTC-TFC2016-master/     OK
    testing: USTC-TFC2016-master/Benign/   OK
    testing: USTC-TFC2016-master/Benign/BitTorrent.pcap   OK
    testing: USTC-TFC2016-master/Benign/FTP.pcap   OK
    testing: USTC-TFC2016-master/Benign/Facetime.pcap   OK
    testing: USTC-TFC2016-master/Benign/Gmail.pcap   OK
    testing: USTC-TFC2016-master/Benign/MySQL.pcap   OK
    testing: USTC-TFC2016-master/Benign/Outlook.pcap   OK
    testing: USTC-TFC2016-master/Benign/SMB.7z   OK
    testing: USTC-TFC2016-master/Benign/Skype.pcap   OK
    testing: USTC-TFC2016-master/Benign/Weibo.7z   OK
    testing: USTC-TFC2016-master/Benign/WorldOfWarcraft.pcap   OK
    testing: USTC-TFC2016-master/LICENSE   OK
    testing: USTC-TFC2016-master/Malware/   OK
    testing: USTC-TFC2016-master/Malware/Cridex.7z   OK
    testing: USTC-TFC2016-master/Malware/Geodo.7z   OK
    testing: USTC-TFC2016-master/Malware/Htbot.7z   O

In [ ]:
import os

extract_path = "/content/USTC-TFC2016"

os.makedirs(extract_path, exist_ok=True)

!unzip -q /content/USTC-TFC2016-master.zip -d /content/USTC-TFC2016

print("✅ Dataset extracted!")

print(os.listdir("/content/USTC-TFC2016"))

replace /content/USTC-TFC2016/USTC-TFC2016-master/Benign/BitTorrent.pcap? [y]es, [n]o, [A]ll, [N]one, [r]ename: ✅ Dataset extracted!
['USTC-TFC2016-master']


In [ ]:
import os

root = "/content/USTC-TFC2016"

print("Root exists:", os.path.exists(root))

for path, dirs, files in os.walk(root):
    pcaps = [f for f in files if f.lower().endswith(".pcap")]
    if pcaps:
        print("\nFolder:", path)
        print("PCAPs:", len(pcaps))
        print(pcaps[:10])

Root exists: True

Folder: /content/USTC-TFC2016/USTC-TFC2016-master/Benign
PCAPs: 8
['FTP.pcap', 'MySQL.pcap', 'Facetime.pcap', 'Gmail.pcap', 'Skype.pcap', 'WorldOfWarcraft.pcap', 'Outlook.pcap', 'BitTorrent.pcap']

Folder: /content/USTC-TFC2016/USTC-TFC2016-master/Malware
PCAPs: 3
['Tinba.pcap', 'Zeus.pcap', 'Miuref.pcap']


In [ ]:
import glob

pcaps = glob.glob("/content/USTC-TFC2016/**/*.pcap", recursive=True)

print("Total PCAP files found:", len(pcaps))
print("\nFirst 10 PCAP files:")
for p in pcaps[:10]:
    print(p)

Total PCAP files found: 11

First 10 PCAP files:
/content/USTC-TFC2016/USTC-TFC2016-master/Benign/FTP.pcap
/content/USTC-TFC2016/USTC-TFC2016-master/Benign/MySQL.pcap
/content/USTC-TFC2016/USTC-TFC2016-master/Benign/Facetime.pcap
/content/USTC-TFC2016/USTC-TFC2016-master/Benign/Gmail.pcap
/content/USTC-TFC2016/USTC-TFC2016-master/Benign/Skype.pcap
/content/USTC-TFC2016/USTC-TFC2016-master/Benign/WorldOfWarcraft.pcap
/content/USTC-TFC2016/USTC-TFC2016-master/Benign/Outlook.pcap
/content/USTC-TFC2016/USTC-TFC2016-master/Benign/BitTorrent.pcap
/content/USTC-TFC2016/USTC-TFC2016-master/Malware/Tinba.pcap
/content/USTC-TFC2016/USTC-TFC2016-master/Malware/Zeus.pcap


In [ ]:
# ==================================================
# VERIFY DATASET STRUCTURE
# ==================================================

benign_pcaps = [
    p for p in all_pcaps
    if (
        "/Benign/" in p
        or "/Benign/" in p.replace(
            "/USTC_extracted/",
            "/"
        )
    )
]

malware_pcaps = [
    p for p in all_pcaps
    if (
        "/Malware/" in p
    )
]

# Better direct check using source folders
benign_pcaps = [
    p for p in all_pcaps
    if "/Benign/" in p
]

malware_pcaps = [
    p for p in all_pcaps
    if "/Malware/" in p
]


# Extracted files have structure:
# /USTC_extracted/Benign/...
# /USTC_extracted/Malware/...

benign_pcaps = [
    p for p in all_pcaps
    if (
        "/Benign/" in p
        or "/USTC_extracted/Benign/" in p
    )
]

malware_pcaps = [
    p for p in all_pcaps
    if (
        "/Malware/" in p
        or "/USTC_extracted/Malware/" in p
    )
]


print(
    "Benign PCAPs :",
    len(benign_pcaps)
)

print(
    "Malware PCAPs:",
    len(malware_pcaps)
)

print(
    "Total PCAPs  :",
    len(all_pcaps)
)


print("\n--- BENIGN ---")

for p in benign_pcaps:
    print(os.path.basename(p))


print("\n--- MALWARE ---")

for p in malware_pcaps:
    print(os.path.basename(p))

Benign PCAPs : 14
Malware PCAPs: 10
Total PCAPs  : 24

--- BENIGN ---
BitTorrent.pcap
FTP.pcap
Facetime.pcap
Gmail.pcap
MySQL.pcap
Outlook.pcap
Skype.pcap
WorldOfWarcraft.pcap
SMB-1.pcap
SMB-2.pcap
Weibo-1.pcap
Weibo-2.pcap
Weibo-3.pcap
Weibo-4.pcap

--- MALWARE ---
Miuref.pcap
Tinba.pcap
Zeus.pcap
Cridex.pcap
Geodo.pcap
Htbot.pcap
Neris.pcap
Nsis-ay.pcap
Shifu.pcap
Virut.pcap


In [ ]:
# ==================================================
# 5. TLS / JA3 / JA3S FEATURE EXTRACTION
# ==================================================

import hashlib

from scapy.layers.tls.record import TLS
from scapy.layers.tls.handshake import (
    TLSClientHello,
    TLSServerHello
)


# ==================================================
# GET LABEL
# ==================================================

def get_label(path):

    path_lower = path.lower()

    if (
        "/benign/" in path_lower
    ):
        return 0

    if (
        "/malware/" in path_lower
    ):
        return 1

    return None


# ==================================================
# CAPTURE ID
# ==================================================

def get_capture_id(path):

    path = os.path.abspath(path)

    # Original dataset
    if "/USTC-TFC2016-master/Benign/" in path:

        relative = path.split(
            "/USTC-TFC2016-master/Benign/",
            1
        )[1]

        return (
            "Benign__" +
            relative.replace("/", "__")
        )

    if "/USTC-TFC2016-master/Malware/" in path:

        relative = path.split(
            "/USTC-TFC2016-master/Malware/",
            1
        )[1]

        return (
            "Malware__" +
            relative.replace("/", "__")
        )

    # Extracted dataset
    if "/USTC_extracted/Benign/" in path:

        relative = path.split(
            "/USTC_extracted/Benign/",
            1
        )[1]

        return (
            "Benign__" +
            relative.replace("/", "__")
        )

    if "/USTC_extracted/Malware/" in path:

        relative = path.split(
            "/USTC_extracted/Malware/",
            1
        )[1]

        return (
            "Malware__" +
            relative.replace("/", "__")
        )

    return os.path.basename(path)


# ==================================================
# MD5
# ==================================================

def md5_hash(value):

    return hashlib.md5(
        value.encode()
    ).hexdigest()


# ==================================================
# JA3 EXTRACTION
# ==================================================

def extract_ja3(client_hello):

    try:

        version = getattr(
            client_hello,
            "version",
            0
        )

        # Cipher suites
        ciphers = []

        cipher_list = getattr(
            client_hello,
            "ciphers",
            []
        )

        for cipher in cipher_list:

            value = getattr(
                cipher,
                "val",
                cipher
            )

            ciphers.append(
                str(value)
            )


        # Extensions
        extensions = []

        ext_list = getattr(
            client_hello,
            "ext",
            []
        )

        for ext in ext_list:

            ext_type = getattr(
                ext,
                "type",
                None
            )

            if ext_type is not None:

                extensions.append(
                    str(ext_type)
                )


        # Supported groups
        groups = []

        for ext in ext_list:

            group_list = getattr(
                ext,
                "groups",
                None
            )

            if group_list:

                for group in group_list:

                    value = getattr(
                        group,
                        "val",
                        group
                    )

                    groups.append(
                        str(value)
                    )


        # EC point formats
        ec_formats = []

        for ext in ext_list:

            format_list = getattr(
                ext,
                "ecpl",
                None
            )

            if format_list:

                for fmt in format_list:

                    value = getattr(
                        fmt,
                        "val",
                        fmt
                    )

                    ec_formats.append(
                        str(value)
                    )


        ja3_string = ",".join([
            str(version),
            "-".join(ciphers),
            "-".join(extensions),
            "-".join(groups),
            "-".join(ec_formats)
        ])


        return (
            ja3_string,
            md5_hash(ja3_string)
        )

    except Exception:

        return "", ""


# ==================================================
# JA3S EXTRACTION
# ==================================================

def extract_ja3s(server_hello):

    try:

        version = getattr(
            server_hello,
            "version",
            0
        )

        cipher = getattr(
            server_hello,
            "cipher",
            ""
        )

        extensions = []

        ext_list = getattr(
            server_hello,
            "ext",
            []
        )

        for ext in ext_list:

            ext_type = getattr(
                ext,
                "type",
                None
            )

            if ext_type is not None:

                extensions.append(
                    str(ext_type)
                )


        ja3s_string = ",".join([
            str(version),
            str(cipher),
            "-".join(extensions)
        ])


        return (
            ja3s_string,
            md5_hash(ja3s_string)
        )

    except Exception:

        return "", ""


# ==================================================
# FLOW FEATURE EXTRACTION
# ==================================================

def extract_pcap_features(pcap_path):

    label = get_label(
        pcap_path
    )

    if label is None:

        return []


    capture_id = get_capture_id(
        pcap_path
    )


    flows = defaultdict(list)


    try:

        reader = PcapReader(
            pcap_path
        )


        for packet in reader:

            # ------------------------------
            # IP
            # ------------------------------

            if IP in packet:

                src = packet[IP].src
                dst = packet[IP].dst

            elif IPv6 in packet:

                src = packet[IPv6].src
                dst = packet[IPv6].dst

            else:

                continue


            # ------------------------------
            # TCP / UDP
            # ------------------------------

            if TCP in packet:

                sport = packet[TCP].sport
                dport = packet[TCP].dport

                protocol = "TCP"

            elif UDP in packet:

                sport = packet[UDP].sport
                dport = packet[UDP].dport

                protocol = "UDP"

            else:

                continue


            # ------------------------------
            # Directional 5-tuple
            # ------------------------------

            flow_key = (
                src,
                sport,
                dst,
                dport,
                protocol
            )


            timestamp = float(
                packet.time
            )

            packet_size = len(
                packet
            )


            # ==================================================
            # TLS METADATA
            # ==================================================

            tls_present = 0

            server_hello = 0

            tls_version = 0

            ja3 = ""

            ja3_hash = ""

            ja3s = ""

            ja3s_hash = ""


            # Actual TLS detection
            if packet.haslayer(TLS):

                tls_present = 1


                # ------------------------------
                # Client Hello / JA3
                # ------------------------------

                if packet.haslayer(
                    TLSClientHello
                ):

                    client_hello = packet[
                        TLSClientHello
                    ]


                    (
                        ja3,
                        ja3_hash
                    ) = extract_ja3(
                        client_hello
                    )


                    tls_version = getattr(
                        client_hello,
                        "version",
                        0
                    )


                # ------------------------------
                # Server Hello / JA3S
                # ------------------------------

                if packet.haslayer(
                    TLSServerHello
                ):

                    server_hello = 1


                    server_hello_layer = packet[
                        TLSServerHello
                    ]


                    (
                        ja3s,
                        ja3s_hash
                    ) = extract_ja3s(
                        server_hello_layer
                    )


                    if tls_version == 0:

                        tls_version = getattr(
                            server_hello_layer,
                            "version",
                            0
                        )


            # Store packet-level data
            flows[
                flow_key
            ].append({

                "timestamp":
                    timestamp,

                "size":
                    packet_size,

                "tls_present":
                    tls_present,

                "server_hello":
                    server_hello,

                "tls_version":
                    tls_version,

                "ja3":
                    ja3,

                "ja3_hash":
                    ja3_hash,

                "ja3s":
                    ja3s,

                "ja3s_hash":
                    ja3s_hash
            })


        reader.close()


    except Exception as e:

        print(
            "Error:",
            pcap_path,
            e
        )

        return []


    # ==================================================
    # CREATE FLOW FEATURES
    # ==================================================

    rows = []


    for flow_key, packets in flows.items():

        src, sport, dst, dport, protocol = flow_key


        packets = sorted(
            packets,
            key=lambda x:
            x["timestamp"]
        )


        times = np.array([
            x["timestamp"]
            for x in packets
        ])


        sizes = np.array([
            x["size"]
            for x in packets
        ])


        # ==================================================
        # BASIC FEATURES
        # ==================================================

        duration = max(
            times[-1] - times[0],
            0.000001
        )


        packet_count = len(
            packets
        )


        byte_count = np.sum(
            sizes
        )


        packet_rate = (
            packet_count /
            duration
        )


        byte_rate = (
            byte_count /
            duration
        )


        # ==================================================
        # BEHAVIOURAL FEATURES
        # ==================================================

        if len(times) > 1:

            iat = np.diff(
                times
            )


            iat_mean = np.mean(
                iat
            )

            iat_std = np.std(
                iat
            )

            iat_max = np.max(
                iat
            )


            iat_cv = (

                iat_std / iat_mean

                if iat_mean > 0

                else 0

            )


            if len(iat) > 2:

                centered = (
                    iat -
                    np.mean(iat)
                )


                denominator = np.sum(
                    centered ** 2
                )


                if denominator > 0:

                    periodicity = (
                        np.sum(
                            centered[:-1]
                            *
                            centered[1:]
                        )
                        /
                        denominator
                    )

                else:

                    periodicity = 0

            else:

                periodicity = 0

        else:

            iat_mean = 0
            iat_std = 0
            iat_max = 0
            iat_cv = 0
            periodicity = 0


        # ==================================================
        # STRUCTURAL FEATURES
        # ==================================================

        packet_mean = np.mean(
            sizes
        )


        packet_std = np.std(
            sizes
        )


        packet_cv = (

            packet_std /
            packet_mean

            if packet_mean > 0

            else 0

        )


        # ==================================================
        # GET FLOW TLS VALUES
        # ==================================================

        ja3_values = [
            x["ja3"]
            for x in packets
            if x["ja3"]
        ]


        ja3_hash_values = [
            x["ja3_hash"]
            for x in packets
            if x["ja3_hash"]
        ]


        ja3s_values = [
            x["ja3s"]
            for x in packets
            if x["ja3s"]
        ]


        ja3s_hash_values = [
            x["ja3s_hash"]
            for x in packets
            if x["ja3s_hash"]
        ]


        tls_versions = [
            x["tls_version"]
            for x in packets
            if x["tls_version"] != 0
        ]


        ja3_value = (
            ja3_values[0]
            if ja3_values
            else ""
        )


        ja3_hash_value = (
            ja3_hash_values[0]
            if ja3_hash_values
            else ""
        )


        ja3s_value = (
            ja3s_values[0]
            if ja3s_values
            else ""
        )


        ja3s_hash_value = (
            ja3s_hash_values[0]
            if ja3s_hash_values
            else ""
        )


        tls_version_value = (

            tls_versions[0]

            if tls_versions

            else 0

        )


        tls_present_flow = int(
            any(
                x["tls_present"]
                for x in packets
            )
        )


        server_hello_flow = int(
            any(
                x["server_hello"]
                for x in packets
            )
        )


        # ==================================================
        # FINAL ROW
        # ==================================================

        rows.append({

            "src_ip":
                src,

            "src_port":
                sport,

            "dst_ip":
                dst,

            "dst_port":
                dport,

            "protocol":
                protocol,


            # Basic
            "duration":
                duration,

            "packet_count":
                packet_count,

            "byte_count":
                byte_count,

            "packet_rate":
                packet_rate,

            "byte_rate":
                byte_rate,


            # Structural
            "packet_size_mean":
                packet_mean,

            "packet_size_std":
                packet_std,

            "packet_size_cv":
                packet_cv,

            "first_packet_size":
                sizes[0],

            "last_packet_size":
                sizes[-1],


            # TLS
            "tls_present":
                tls_present_flow,

            "tls_version":
                tls_version_value,

            "server_hello":
                server_hello_flow,


            # JA3 / JA3S
            "ja3":
                ja3_value,

            "ja3_hash":
                ja3_hash_value,

            "ja3s":
                ja3s_value,

            "ja3s_hash":
                ja3s_hash_value,


            # Behavioural
            "iat_mean":
                iat_mean,

            "iat_std":
                iat_std,

            "iat_max":
                iat_max,

            "iat_cv":
                iat_cv,

            "periodicity_score":
                periodicity,


            # Capture
            "capture_id":
                capture_id,


            # Target
            "label":
                label
        })


    return rows

In [ ]:
# ==================================================
# 3. FIND ALL PCAP FILES
# ==================================================

import os
import glob

DATASET_PATH = "/content/USTC-TFC2016/USTC-TFC2016-master"
EXTRACT_ROOT = "/content/USTC_extracted"

all_pcaps = []

# Original PCAPs
all_pcaps += glob.glob(
    os.path.join(DATASET_PATH, "**", "*.pcap"),
    recursive=True
)

# PCAPs extracted from .7z files
all_pcaps += glob.glob(
    os.path.join(EXTRACT_ROOT, "**", "*.pcap"),
    recursive=True
)

# Remove duplicate paths
all_pcaps = sorted(
    set(os.path.abspath(p) for p in all_pcaps)
)

print("Total PCAP files found:", len(all_pcaps))

for p in all_pcaps:
    print(p)

Total PCAP files found: 24
/content/USTC-TFC2016/USTC-TFC2016-master/Benign/BitTorrent.pcap
/content/USTC-TFC2016/USTC-TFC2016-master/Benign/FTP.pcap
/content/USTC-TFC2016/USTC-TFC2016-master/Benign/Facetime.pcap
/content/USTC-TFC2016/USTC-TFC2016-master/Benign/Gmail.pcap
/content/USTC-TFC2016/USTC-TFC2016-master/Benign/MySQL.pcap
/content/USTC-TFC2016/USTC-TFC2016-master/Benign/Outlook.pcap
/content/USTC-TFC2016/USTC-TFC2016-master/Benign/Skype.pcap
/content/USTC-TFC2016/USTC-TFC2016-master/Benign/WorldOfWarcraft.pcap
/content/USTC-TFC2016/USTC-TFC2016-master/Malware/Miuref.pcap
/content/USTC-TFC2016/USTC-TFC2016-master/Malware/Tinba.pcap
/content/USTC-TFC2016/USTC-TFC2016-master/Malware/Zeus.pcap
/content/USTC_extracted/Benign/SMB/SMB/SMB-1.pcap
/content/USTC_extracted/Benign/SMB/SMB/SMB-2.pcap
/content/USTC_extracted/Benign/Weibo/Weibo/Weibo-1.pcap
/content/USTC_extracted/Benign/Weibo/Weibo/Weibo-2.pcap
/content/USTC_extracted/Benign/Weibo/Weibo/Weibo-3.pcap
/content/USTC_extracted/

In [ ]:
# ==================================================
# 6. BUILD COMPLETE FEATURE DATASET
# ==================================================

all_features = []

print("Extracting features...")

for i, pcap in enumerate(all_pcaps):

    print(
        f"[{i+1}/{len(all_pcaps)}] "
        f"{os.path.basename(pcap)}"
    )

    features = extract_pcap_features(
        pcap
    )

    all_features.extend(
        features
    )


df = pd.DataFrame(
    all_features
)


print("\nFinal Dataset Shape:")
print(df.shape)

print("\nClass Distribution:")
print(df["label"].value_counts())

df.head()

Extracting features...
[1/24] BitTorrent.pcap
[2/24] FTP.pcap
[3/24] Facetime.pcap
[4/24] Gmail.pcap
[5/24] MySQL.pcap
[6/24] Outlook.pcap
[7/24] Skype.pcap


[8/24] WorldOfWarcraft.pcap
[9/24] Miuref.pcap
[10/24] Tinba.pcap
[11/24] Zeus.pcap
[12/24] SMB-1.pcap
[13/24] SMB-2.pcap
[14/24] Weibo-1.pcap
[15/24] Weibo-2.pcap
[16/24] Weibo-3.pcap
[17/24] Weibo-4.pcap
[18/24] Cridex.pcap
[19/24] Geodo.pcap
[20/24] Htbot.pcap


[21/24] Neris.pcap


[22/24] Nsis-ay.pcap
[23/24] Shifu.pcap
[24/24] Virut.pcap



Final Dataset Shape:
(886288, 29)

Class Distribution:
label
0    612662
1    273626
Name: count, dtype: int64


,src_ip,src_port,dst_ip,dst_port,protocol,duration,packet_count,byte_count,packet_rate,byte_rate,...,ja3_hash,ja3s,ja3s_hash,iat_mean,iat_std,iat_max,iat_cv,periodicity_score,capture_id,label
0,1.1.33.158,41319,1.2.156.163,443,TCP,0.000001,1,1475,1000000.0,1.475000e+09,...,,,,0.0,0.0,0.0,0.0,0.0,Benign__BitTorrent.pcap,0
1,1.1.23.218,9341,1.2.31.193,443,TCP,0.000001,1,163,1000000.0,1.630000e+08,...,,,,0.0,0.0,0.0,0.0,0.0,Benign__BitTorrent.pcap,0
2,1.2.84.77,443,1.1.252.248,22794,TCP,0.000001,1,70,1000000.0,7.000000e+07,...,,,,0.0,0.0,0.0,0.0,0.0,Benign__BitTorrent.pcap,0
3,1.1.71.180,65486,1.2.79.35,443,TCP,0.000001,1,1475,1000000.0,1.475000e+09,...,,,,0.0,0.0,0.0,0.0,0.0,Benign__BitTorrent.pcap,0
4,1.1.41.222,14880,1.2.129.94,443,TCP,0.000001,1,136,1000000.0,1.360000e+08,...,,,,0.0,0.0,0.0,0.0,0.0,Benign__BitTorrent.pcap,0


In [ ]:
#==================================================
# 7. DATA CLEANING + FEATURE GROUPS
# ==================================================

print("Original Shape:", df.shape)

# Replace infinite values
df = df.replace(
    [np.inf, -np.inf],
    np.nan
)

# Remove duplicate rows
df = df.drop_duplicates()

# Remove rows with missing values
df = df.dropna()

print("After Cleaning:", df.shape)


# ==================================================
# 8. STRUCTURAL FEATURES
# ==================================================

structural_features = [

    "packet_size_mean",
    "packet_size_std",
    "packet_size_cv",
    "first_packet_size",
    "last_packet_size",

    # TLS metadata
    "tls_present",
    "tls_version",
    "server_hello",

    # TLS fingerprints
    "ja3_hash_freq",
    "ja3s_hash_freq"
]

# ==================================================
# 9. BEHAVIOURAL FEATURES
# ==================================================

behavioural_features = [

    "duration",
    "packet_count",
    "byte_count",
    "packet_rate",
    "byte_rate",

    "iat_mean",
    "iat_std",
    "iat_max",
    "iat_cv",

    # Periodicity belongs to behavioural detector
    "periodicity_score"
]


print("\nStructural features:")
print(structural_features)

print("\nBehavioural features:")
print(behavioural_features)

print("\nFeature groups created successfully.")

Original Shape: (886288, 29)
After Cleaning: (886288, 29)

Structural features:
['packet_size_mean', 'packet_size_std', 'packet_size_cv', 'first_packet_size', 'last_packet_size', 'tls_present', 'tls_version', 'server_hello', 'ja3_hash_freq', 'ja3s_hash_freq']

Behavioural features:
['duration', 'packet_count', 'byte_count', 'packet_rate', 'byte_rate', 'iat_mean', 'iat_std', 'iat_max', 'iat_cv', 'periodicity_score']

Feature groups created successfully.


In [ ]:
# ==================================================
# 10. STRATIFIED CAPTURE-WISE TRAIN / VAL / TEST SPLIT
# ==================================================

from sklearn.model_selection import train_test_split

# --------------------------------------------------
# Remove old leakage-prone detector scores
# --------------------------------------------------

df = df.drop(
    columns=[
        "structural_score",
        "behavioural_score"
    ],
    errors="ignore"
)


# --------------------------------------------------
# Get unique captures and their labels
# --------------------------------------------------

capture_labels = (
    df[
        ["capture_id", "label"]
    ]
    .drop_duplicates()
)


# Separate benign and malware captures
benign_captures = capture_labels[
    capture_labels["label"] == 0
]["capture_id"].tolist()

malware_captures = capture_labels[
    capture_labels["label"] == 1
]["capture_id"].tolist()


print("Benign captures :", len(benign_captures))
print("Malware captures:", len(malware_captures))


# --------------------------------------------------
# Split each class separately
# 60% Train / 20% Validation / 20% Test
# --------------------------------------------------

# ---------- BENIGN ----------

benign_train, benign_temp = train_test_split(
    benign_captures,
    test_size=0.40,
    random_state=RANDOM_SEED
)

benign_val, benign_test = train_test_split(
    benign_temp,
    test_size=0.50,
    random_state=RANDOM_SEED
)


# ---------- MALWARE ----------

malware_train, malware_temp = train_test_split(
    malware_captures,
    test_size=0.40,
    random_state=RANDOM_SEED
)

malware_val, malware_test = train_test_split(
    malware_temp,
    test_size=0.50,
    random_state=RANDOM_SEED
)


# --------------------------------------------------
# Combine capture IDs
# --------------------------------------------------

train_captures = benign_train + malware_train
val_captures = benign_val + malware_val
test_captures = benign_test + malware_test


# --------------------------------------------------
# Create datasets using capture IDs
# --------------------------------------------------

train_df = df[
    df["capture_id"].isin(train_captures)
].copy()

val_df = df[
    df["capture_id"].isin(val_captures)
].copy()

test_df = df[
    df["capture_id"].isin(test_captures)
].copy()


# --------------------------------------------------
# Check results
# --------------------------------------------------

print("\nTraining   :", train_df.shape)
print("Validation :", val_df.shape)
print("Testing    :", test_df.shape)

print("\nTraining labels:")
print(train_df["label"].value_counts())

print("\nValidation labels:")
print(val_df["label"].value_counts())

print("\nTesting labels:")
print(test_df["label"].value_counts())


# ==================================================
# 11. FEATURES
# ==================================================

drop_columns = [

    "label",
    "capture_id",

    # Network identifiers
    "src_ip",
    "src_port",
    "dst_ip",
    "dst_port",

    # Raw JA3 strings
    "ja3",
    "ja3s"
]


X_train = train_df.drop(
    columns=drop_columns,
    errors="ignore"
)

y_train = train_df["label"]


X_val = val_df.drop(
    columns=drop_columns,
    errors="ignore"
)

y_val = val_df["label"]


X_test = test_df.drop(
    columns=drop_columns,
    errors="ignore"
)

y_test = test_df["label"]


# --------------------------------------------------
# Convert protocol to numeric
# --------------------------------------------------

for data in [
    X_train,
    X_val,
    X_test
]:

    data["protocol"] = (
        data["protocol"]
        .map({
            "TCP": 0,
            "UDP": 1
        })
        .fillna(-1)
    )


print("\nTraining features:")
print(X_train.columns.tolist())

print(
    "\nNumber of training features:",
    X_train.shape[1]
)

Benign captures : 14
Malware captures: 10

Training   : (693041, 29)
Validation : (68436, 29)
Testing    : (124811, 29)

Training labels:
label
0    484796
1    208245
Name: count, dtype: int64

Validation labels:
label
0    40018
1    28418
Name: count, dtype: int64

Testing labels:
label
0    87848
1    36963
Name: count, dtype: int64

Training features:
['protocol', 'duration', 'packet_count', 'byte_count', 'packet_rate', 'byte_rate', 'packet_size_mean', 'packet_size_std', 'packet_size_cv', 'first_packet_size', 'last_packet_size', 'tls_present', 'tls_version', 'server_hello', 'ja3_hash', 'ja3s_hash', 'iat_mean', 'iat_std', 'iat_max', 'iat_cv', 'periodicity_score']

Number of training features: 21


In [ ]:
# ==================================================
# 12. JA3 / JA3S FREQUENCY ENCODING
# ==================================================

def add_frequency_encoding(
    X_train,
    X_val,
    X_test,
    column
):

    train_frequency = (
        X_train[column]
        .value_counts()
    )


    X_train[
        column + "_freq"
    ] = (
        X_train[column]
        .map(train_frequency)
        .fillna(0)
    )


    X_val[
        column + "_freq"
    ] = (
        X_val[column]
        .map(train_frequency)
        .fillna(0)
    )


    X_test[
        column + "_freq"
    ] = (
        X_test[column]
        .map(train_frequency)
        .fillna(0)
    )


    return (
        X_train,
        X_val,
        X_test
    )


# JA3
(
    X_train,
    X_val,
    X_test
) = add_frequency_encoding(
    X_train,
    X_val,
    X_test,
    "ja3_hash"
)


# JA3S
(
    X_train,
    X_val,
    X_test
) = add_frequency_encoding(
    X_train,
    X_val,
    X_test,
    "ja3s_hash"
)


# Remove raw hashes after encoding
X_train = X_train.drop(
    columns=[
        "ja3_hash",
        "ja3s_hash"
    ]
)

X_val = X_val.drop(
    columns=[
        "ja3_hash",
        "ja3s_hash"
    ]
)

X_test = X_test.drop(
    columns=[
        "ja3_hash",
        "ja3s_hash"
    ]
)


print(
    "Final features:"
)

print(
    X_train.columns.tolist()
)

Final features:
['protocol', 'duration', 'packet_count', 'byte_count', 'packet_rate', 'byte_rate', 'packet_size_mean', 'packet_size_std', 'packet_size_cv', 'first_packet_size', 'last_packet_size', 'tls_present', 'tls_version', 'server_hello', 'iat_mean', 'iat_std', 'iat_max', 'iat_cv', 'periodicity_score', 'ja3_hash_freq', 'ja3s_hash_freq']


In [ ]:
# ==================================================
# 12. MODEL TRAINING
# 13. MODEL COMPARISON
# ==================================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

models = {}


# ==================================================
# RANDOM FOREST
# ==================================================

models["Random Forest"] = RandomForestClassifier(

    n_estimators=300,

    max_depth=20,

    min_samples_split=5,

    min_samples_leaf=4,

    max_features="sqrt",

    class_weight="balanced_subsample",

    random_state=RANDOM_SEED,

    n_jobs=-1
)


# ==================================================
# XGBOOST
# ==================================================

if HAS_XGB:

    negative = (y_train == 0).sum()
    positive = (y_train == 1).sum()

    scale_pos_weight = negative / positive

    models["XGBoost"] = xgb.XGBClassifier(

        n_estimators=300,

        max_depth=5,

        learning_rate=0.05,

        min_child_weight=5,

        subsample=0.8,

        colsample_bytree=0.8,

        gamma=0.1,

        reg_alpha=0.1,

        reg_lambda=1.0,

        scale_pos_weight=scale_pos_weight,

        objective="binary:logistic",

        eval_metric="logloss",

        random_state=RANDOM_SEED,

        n_jobs=-1
    )


# ==================================================
# LIGHTGBM
# ==================================================

if HAS_LGB:

    models["LightGBM"] = lgb.LGBMClassifier(

        n_estimators=300,

        num_leaves=31,

        max_depth=8,

        learning_rate=0.05,

        min_child_samples=30,

        subsample=0.8,

        colsample_bytree=0.8,

        reg_alpha=0.1,

        reg_lambda=1.0,

        class_weight="balanced",

        random_state=RANDOM_SEED,

        verbosity=-1,

        n_jobs=-1
    )


# ==================================================
# FIND BEST THRESHOLD
# ==================================================

def find_best_threshold(y_true, probabilities):

    thresholds = np.arange(
        0.10,
        0.91,
        0.01
    )

    best_threshold = 0.50
    best_f1 = 0.0

    for threshold in thresholds:

        predictions = (
            probabilities >= threshold
        ).astype(int)

        score = f1_score(
            y_true,
            predictions,
            zero_division=0
        )

        if score > best_f1:

            best_f1 = score
            best_threshold = threshold

    return best_threshold, best_f1


# ==================================================
# TRAIN MODELS
# ==================================================

best_estimators = {}
results = {}

print("Training models...\n")


for name, model in models.items():

    print(f"Training {name}...")

    start_time = time.time()

    # ------------------------------------------------
    # TRAIN
    # ------------------------------------------------

    model.fit(
        X_train,
        y_train
    )

    training_time = time.time() - start_time

    best_estimators[name] = model


    # ------------------------------------------------
    # VALIDATION PROBABILITY
    # ------------------------------------------------

    y_val_prob = model.predict_proba(
        X_val
    )[:, 1]


    # ------------------------------------------------
    # BEST THRESHOLD
    # ------------------------------------------------

    best_threshold, best_f1 = find_best_threshold(
        y_val,
        y_val_prob
    )


    # ------------------------------------------------
    # VALIDATION PREDICTION
    # ------------------------------------------------

    y_val_pred = (
        y_val_prob >= best_threshold
    ).astype(int)


    # ------------------------------------------------
    # METRICS
    # ------------------------------------------------

    results[name] = {

        "Model": name,

        "Threshold": best_threshold,

        "Accuracy": accuracy_score(
            y_val,
            y_val_pred
        ),

        "Precision": precision_score(
            y_val,
            y_val_pred,
            zero_division=0
        ),

        "Recall": recall_score(
            y_val,
            y_val_pred,
            zero_division=0
        ),

        "F1": f1_score(
            y_val,
            y_val_pred,
            zero_division=0
        ),

        "ROC-AUC": roc_auc_score(
            y_val,
            y_val_prob
        ),

        "Training Time (sec)": training_time
    }


    print(
        f"Completed in {training_time:.2f} sec"
    )

    print(
        f"Threshold : {best_threshold:.2f}"
    )

    print(
        f"Precision : "
        f"{results[name]['Precision']:.4f}"
    )

    print(
        f"Recall    : "
        f"{results[name]['Recall']:.4f}"
    )

    print(
        f"F1        : "
        f"{results[name]['F1']:.4f}"
    )

    print(
        f"ROC-AUC   : "
        f"{results[name]['ROC-AUC']:.4f}\n"
    )


# ==================================================
# MODEL COMPARISON
# ==================================================

results_df = (
    pd.DataFrame(
        list(results.values())
    )
    .sort_values(
        by="F1",
        ascending=False
    )
    .reset_index(drop=True)
)


print("\n==========================================")
print("VALIDATION PERFORMANCE")
print("==========================================")

print(
    results_df.to_string(
        index=False
    )
)


# ==================================================
# SAVE COMPARISON
# ==================================================

results_df.to_csv(
    "module5_model_comparison.csv",
    index=False
)


# ==================================================
# SELECT BEST MODEL
# ==================================================

best_model_name = results_df.iloc[0]["Model"]

best_model = best_estimators[
    best_model_name
]

best_threshold = results_df.iloc[0]["Threshold"]


print("\n==========================================")
print("SELECTED MODEL")
print("==========================================")

print("Model     :", best_model_name)
print("Threshold :", best_threshold)

Training models...

Training Random Forest...
Completed in 193.29 sec
Threshold : 0.15
Precision : 1.0000
Recall    : 0.9869
F1        : 0.9934
ROC-AUC   : 0.9990

Training XGBoost...
Completed in 22.11 sec
Threshold : 0.10
Precision : 1.0000
Recall    : 0.9867
F1        : 0.9933
ROC-AUC   : 1.0000

Training LightGBM...
Completed in 17.86 sec
Threshold : 0.10
Precision : 1.0000
Recall    : 0.9867
F1        : 0.9933
ROC-AUC   : 1.0000


VALIDATION PERFORMANCE
        Model  Threshold  Accuracy  Precision   Recall       F1  ROC-AUC  Training Time (sec)
Random Forest       0.15  0.994564        1.0 0.986910 0.993412 0.998979           193.285912
     LightGBM       0.10  0.994491        1.0 0.986734 0.993323 0.999996            17.860773
      XGBoost       0.10  0.994462        1.0 0.986663 0.993287 0.999994            22.110350

SELECTED MODEL
Model     : Random Forest
Threshold : 0.14999999999999997


In [ ]:
# ==================================================
# 15. FINAL TEST EVALUATION
# ==================================================

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# --------------------------------------------------
# Test probabilities
# --------------------------------------------------

y_test_prob = (
    best_model
    .predict_proba(X_test)[:, 1]
)

# Use threshold selected on validation set
y_test_pred = (
    y_test_prob >= best_threshold
).astype(int)


# ==================================================
# FINAL METRICS
# ==================================================

test_accuracy = accuracy_score(
    y_test,
    y_test_pred
)

test_precision = precision_score(
    y_test,
    y_test_pred,
    zero_division=0
)

test_recall = recall_score(
    y_test,
    y_test_pred,
    zero_division=0
)

test_f1 = f1_score(
    y_test,
    y_test_pred,
    zero_division=0
)

test_roc_auc = roc_auc_score(
    y_test,
    y_test_prob
)


# ==================================================
# PRINT RESULTS
# ==================================================

print("\n==========================================")
print("FINAL TEST PERFORMANCE")
print("==========================================")

print(
    f"Model     : {best_model_name}"
)

print(
    f"Threshold : {best_threshold:.2f}"
)

print(
    f"Accuracy  : {test_accuracy:.4f}"
)

print(
    f"Precision : {test_precision:.4f}"
)

print(
    f"Recall    : {test_recall:.4f}"
)

print(
    f"F1 Score  : {test_f1:.4f}"
)

print(
    f"ROC-AUC   : {test_roc_auc:.4f}"
)


# ==================================================
# CONFUSION MATRIX
# ==================================================

cm = confusion_matrix(
    y_test,
    y_test_pred
)

print("\nConfusion Matrix:")
print(cm)


# ==================================================
# CLASSIFICATION REPORT
# ==================================================

print("\nClassification Report:")

print(
    classification_report(
        y_test,
        y_test_pred,
        target_names=[
            "Benign",
            "Malware"
        ],
        zero_division=0
    )
)


FINAL TEST PERFORMANCE
Model     : Random Forest
Threshold : 0.15
Accuracy  : 0.9921
Precision : 0.9742
Recall    : 1.0000
F1 Score  : 0.9869
ROC-AUC   : 1.0000

Confusion Matrix:
[[86868   980]
 [    0 36963]]

Classification Report:
              precision    recall  f1-score   support

      Benign       1.00      0.99      0.99     87848
     Malware       0.97      1.00      0.99     36963

    accuracy                           0.99    124811
   macro avg       0.99      0.99      0.99    124811
weighted avg       0.99      0.99      0.99    124811



In [ ]:
# ==================================================
# 16. FINAL MODEL SELECTION
# ==================================================

final_model = best_estimators[best_model_name]

print("Final model:", best_model_name)
print("Final threshold:", best_threshold)


# ==================================================
# 17. FEATURE IMPORTANCE
# ==================================================

if hasattr(final_model, "feature_importances_"):

    importance = pd.DataFrame({
        "Feature": X_train.columns,
        "Importance": final_model.feature_importances_
    })

    importance = (
        importance
        .sort_values(
            "Importance",
            ascending=False
        )
        .reset_index(drop=True)
    )

    print("\nTop 20 Important Features:")
    print(importance.head(20))


    plt.figure(figsize=(10, 6))

    sns.barplot(
        data=importance.head(15),
        x="Importance",
        y="Feature"
    )

    plt.title(
        f"Feature Importance - {best_model_name}"
    )

    plt.tight_layout()
    plt.show()


# ==================================================
# 18. SAVE FINAL MODEL
# ==================================================

model_artifact = {

    "model": final_model,

    "features":
        X_train.columns.tolist(),

    "structural_features":
        structural_features,

    "behavioural_features":
        behavioural_features,

    "model_name":
        best_model_name,

    "threshold":
        best_threshold
}


joblib.dump(
    model_artifact,
    "final_module5_malware_model.joblib"
)


print("\nModel saved successfully.")
print("File: final_module5_malware_model.joblib")

NameError: name 'best_estimators' is not defined